In [12]:
!pip install nibabel numpy pandas matplotlib tqdm


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
from pathlib import Path

# DATA_ROOT = Path(r"C:/Users/whate/Documents/Brain CT Hemorrage Detection/Dataset/al_pool/train_2")
DATA_ROOT = Path(r"C:/Users/whate/Documents/Brain CT Hemorrage Detection/Dataset/labeled")

IMAGE_DIR = DATA_ROOT / "ground_truths"
MASK_DIR = DATA_ROOT / "images"

OUTPUT_DIR = DATA_ROOT / "analysis_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset exists:", DATA_ROOT.exists())
print("Images directory exists:", IMAGE_DIR.exists())
print("Masks directory exists:", MASK_DIR.exists())
print("Output directory:", OUTPUT_DIR)

for path in DATA_ROOT.rglob("*"):
    if path.is_file() and (
        path.name.endswith(".nii.gz") or path.name.endswith(".nii")
    ):
        print(path.relative_to(DATA_ROOT))

Dataset exists: True
Images directory exists: True
Masks directory exists: True
Output directory: C:\Users\whate\Documents\Brain CT Hemorrage Detection\Dataset\labeled\analysis_outputs
ground_truths\ID_0237f3c9_ID_40015688b9.nii.gz
ground_truths\ID_029ebbea_ID_563afa0901.nii.gz
ground_truths\ID_02b882cc_ID_a4892e60ae.nii.gz
ground_truths\ID_02f779fb_ID_c4d7f33559.nii.gz
ground_truths\ID_041e6601_ID_1fa1d20bba.nii.gz
ground_truths\ID_0492041f_ID_b4fcf2f799.nii.gz
ground_truths\ID_05331dc6_ID_47213b7e2e.nii.gz
ground_truths\ID_07365cde_ID_8e661b8f07.nii.gz
ground_truths\ID_089596a9_ID_ea71b57820.nii.gz
ground_truths\ID_09b33fc4_ID_bcd322d4ed.nii.gz
ground_truths\ID_0b10cbee_ID_f91d6a7cd2.nii.gz
ground_truths\ID_0c3eef60_ID_6994ad7df0.nii.gz
ground_truths\ID_0c6b97a9_ID_6011bb9ce8.nii.gz
ground_truths\ID_10beb45f_ID_7184e80f09.nii.gz
ground_truths\ID_10c64d22_ID_4c9aad31ca.nii.gz
ground_truths\ID_14058625_ID_527c581583.nii.gz
ground_truths\ID_1653b22b_ID_c35cd0de73.nii.gz
ground_truths\ID

In [14]:

from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib
matplotlib.use("Agg")  # Save plots without requiring a GUI
import matplotlib.pyplot as plt
from tqdm import tqdm

# -------- CONFIGURATION --------
# DATA_ROOT = Path(r"D:\Datasets\INSTANCE2022")
# IMAGE_DIR = DATA_ROOT / "images"
# MASK_DIR = DATA_ROOT / "masks"
# OUTPUT_DIR = DATA_ROOT / "analysis_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find_nifti_files(directory):
    if not directory.exists():
        return []
    return sorted(
        p for p in directory.rglob("*")
        if p.is_file()
        and (p.name.endswith(".nii.gz") or p.name.endswith(".nii"))
    )


def get_stem(path):
    name = path.name
    return name[:-7] if name.endswith(".nii.gz") else name[:-4]


images = {get_stem(p): p for p in find_nifti_files(IMAGE_DIR)}
masks = {get_stem(p): p for p in find_nifti_files(MASK_DIR)}

common = sorted(images.keys() & masks.keys())
print(f"Images: {len(images)}")
print(f"Masks: {len(masks)}")
print(f"Matched pairs: {len(common)}")
print(f"Unmatched images: {len(images.keys() - masks.keys())}")
print(f"Unmatched masks: {len(masks.keys() - images.keys())}")

volume_rows = []
label_rows = []
slice_rows = []
errors = []

for study_id in tqdm(common, desc="CPU dataset audit"):
    try:
        img_nii = nib.load(str(images[study_id]))
        mask_nii = nib.load(str(masks[study_id]))

        # Proxy-backed access: avoid loading entire datasets at once.
        img = img_nii.dataobj
        msk = mask_nii.dataobj

        if len(img_nii.shape) != 3 or len(mask_nii.shape) != 3:
            errors.append({
                "study_id": study_id,
                "issue": "Image or mask is not 3D"
            })
            continue

        image_shape = tuple(img_nii.shape)
        mask_shape = tuple(mask_nii.shape)
        shape_match = image_shape == mask_shape
        affine_match = np.allclose(
            img_nii.affine, mask_nii.affine, atol=1e-4
        )

        # Load one volume at a time.
        image = np.asarray(img, dtype=np.float32)
        mask = np.asarray(msk)

        if not np.isfinite(image).all():
            errors.append({
                "study_id": study_id,
                "issue": "Image has NaN or infinite values"
            })
            continue

        if not np.isfinite(mask).all():
            errors.append({
                "study_id": study_id,
                "issue": "Mask has NaN or infinite values"
            })
            continue

        labels, counts = np.unique(mask, return_counts=True)
        total = mask.size
        foreground = int(np.count_nonzero(mask != 0))

        volume_rows.append({
            "study_id": study_id,
            "image_path": str(images[study_id]),
            "mask_path": str(masks[study_id]),
            "image_shape": str(image_shape),
            "mask_shape": str(mask_shape),
            "image_dtype": str(img_nii.get_data_dtype()),
            "mask_dtype": str(mask_nii.get_data_dtype()),
            "spacing_x_mm": img_nii.header.get_zooms()[0],
            "spacing_y_mm": img_nii.header.get_zooms()[1],
            "spacing_z_mm": img_nii.header.get_zooms()[2],
            "orientation": str(nib.aff2axcodes(img_nii.affine)),
            "shape_match": shape_match,
            "affine_match": affine_match,
            "image_min": float(image.min()),
            "image_max": float(image.max()),
            "image_mean": float(image.mean()),
            "image_std": float(image.std()),
            "total_voxels": int(total),
            "foreground_voxels": foreground,
            "foreground_percent": 100 * foreground / total,
            "labels": str(labels.tolist()),
        })

        for label, count in zip(labels, counts):
            label_rows.append({
                "study_id": study_id,
                "label": label.item(),
                "voxel_count": int(count),
                "voxel_percent": 100 * count / total,
            })

        # Count foreground per slice without retaining extra volumes.
        for z in range(mask.shape[2]):
            slice_mask = mask[:, :, z]
            slice_rows.append({
                "study_id": study_id,
                "slice_index": z,
                "foreground_voxels": int(np.count_nonzero(slice_mask)),
            })

        if not shape_match:
            errors.append({
                "study_id": study_id,
                "issue": f"Shape mismatch: {image_shape} vs {mask_shape}"
            })

        if not affine_match:
            errors.append({
                "study_id": study_id,
                "issue": "Image and mask affines differ"
            })

        del image, mask

    except Exception as exc:
        errors.append({
            "study_id": study_id,
            "issue": f"{type(exc).__name__}: {exc}"
        })

volumes_df = pd.DataFrame(volume_rows)
labels_df = pd.DataFrame(label_rows)
slices_df = pd.DataFrame(slice_rows)
errors_df = pd.DataFrame(errors)

if not labels_df.empty:
    label_summary = (
        labels_df.groupby("label", as_index=False)["voxel_count"]
        .sum()
        .sort_values("label")
    )
    label_summary["percentage"] = (
        100 * label_summary["voxel_count"]
        / label_summary["voxel_count"].sum()
    )
else:
    label_summary = pd.DataFrame(
        columns=["label", "voxel_count", "percentage"]
    )

reports = {
    "volume_metadata.csv": volumes_df,
    "label_statistics.csv": labels_df,
    "label_summary.csv": label_summary,
    "slice_statistics.csv": slices_df,
    "data_quality_issues.csv": errors_df,
}

for filename, df in reports.items():
    df.to_csv(OUTPUT_DIR / filename, index=False)

print("\n--- AUDIT SUMMARY ---")
print("Successfully audited:", len(volumes_df))
print("Matched input pairs:", len(common))
print("Quality issues:", len(errors_df))

if not volumes_df.empty:
    print("Shape mismatches:", int((~volumes_df["shape_match"]).sum()))
    print("Affine mismatches:", int((~volumes_df["affine_match"]).sum()))
    print("\nVolume metadata:")
    print(volumes_df.describe(include="all").to_string())

print("\nLabel summary:")
print(label_summary.to_string(index=False))

print("\nReports saved to:", OUTPUT_DIR)


Images: 192
Masks: 192
Matched pairs: 192
Unmatched images: 0
Unmatched masks: 0


CPU dataset audit: 100%|██████████| 192/192 [01:06<00:00,  2.89it/s]



--- AUDIT SUMMARY ---
Successfully audited: 192
Matched input pairs: 192
Quality issues: 4
Shape mismatches: 0
Affine mismatches: 4

Volume metadata:
                         study_id                                                                                                            image_path                                                                                                      mask_path     image_shape      mask_shape image_dtype mask_dtype  spacing_x_mm  spacing_y_mm  spacing_z_mm      orientation shape_match affine_match  image_min   image_max  image_mean   image_std  total_voxels  foreground_voxels  foreground_percent                                                                                                                                                                                                                                                                                                                                                            

In [15]:

from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# DATA_ROOT = Path(r"D:\Datasets\INSTANCE2022")
OUT = DATA_ROOT / "analysis_outputs"

labels = pd.read_csv(OUT / "label_summary.csv")
volumes = pd.read_csv(OUT / "volume_metadata.csv")
slices = pd.read_csv(OUT / "slice_statistics.csv")

# 1. Class imbalance
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(labels["label"].astype(str), labels["voxel_count"])
ax.set_yscale("log")
ax.set_xlabel("Label ID")
ax.set_ylabel("Voxel count (log scale)")
ax.set_title("INSTANCE 2022: Label distribution")
fig.tight_layout()
fig.savefig(OUT / "label_distribution.png", dpi=150)
plt.close(fig)

# 2. Volume size
volume_slices = volumes["image_shape"].str.extract(
    r"\((\d+),\s*(\d+),\s*(\d+)\)"
).astype(float)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(volume_slices.iloc[:, 2], bins=20)
ax.set_xlabel("Slices per volume")
ax.set_ylabel("Number of studies")
ax.set_title("Volume depth distribution")
fig.tight_layout()
fig.savefig(OUT / "volume_depth.png", dpi=150)
plt.close(fig)

# 3. Foreground occupancy
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(volumes["foreground_percent"].dropna(), bins=30)
ax.set_xlabel("Foreground voxels (%)")
ax.set_ylabel("Number of studies")
ax.set_title("Foreground proportion per volume")
fig.tight_layout()
fig.savefig(OUT / "foreground_distribution.png", dpi=150)
plt.close(fig)

# 4. Slice-level foreground
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(slices["foreground_voxels"], bins=50)
ax.set_xlabel("Foreground voxels per slice")
ax.set_ylabel("Number of slices")
ax.set_title("Foreground distribution across slices")
fig.tight_layout()
fig.savefig(OUT / "slice_foreground_distribution.png", dpi=150)
plt.close(fig)

print("Plots saved to:", OUT)


Plots saved to: C:\Users\whate\Documents\Brain CT Hemorrage Detection\Dataset\labeled\analysis_outputs
